# Gene annotation with the Genomic Intelligence model package

This notebook deploys the Genomic Intelligence gene annotation model from AWS Marketplace in your own account, annotates the sample sequences on a real-time endpoint (as JSON and as GFF3), runs the same samples as a batch transform job, and deletes everything it created.

**Before you run it**

1. Subscribe to the product on AWS Marketplace (search for *Genomic Intelligence Gene Annotation*), then choose **Continue to configuration**. The notebook finds the model package for your Region itself.
2. Run it from SageMaker Studio or a notebook instance whose execution role can create SageMaker models, endpoints and transform jobs, and read and write the Region's default SageMaker bucket.
3. You need quota for one `ml.g5.xlarge` endpoint and one `ml.g5.xlarge` transform job in your Region.

**Cost.** The endpoint bills from the moment it is `InService` until the cleanup cell deletes it, at the SageMaker instance rate plus the product's software price. Run the cleanup cell even if something fails.

Research use only. Not for diagnostic or clinical use.

In [ ]:
import json, time
from pathlib import Path

import boto3

session = boto3.session.Session()
region = session.region_name
sm = session.client("sagemaker")
runtime = session.client("sagemaker-runtime")
s3 = session.client("s3")
account = session.client("sts").get_caller_identity()["Account"]

try:
    import sagemaker
    role = sagemaker.get_execution_role()
except Exception:
    raise SystemExit("Run this from SageMaker Studio or a notebook instance, or set `role` to an execution role ARN.")

bucket = f"sagemaker-{region}-{account}"
prefix = "gi-annotation-example"
INSTANCE = "ml.g5.xlarge"
REALTIME_MAX_BP = None  # TODO(measure): the real-time limit; larger samples go to batch transform only
print(region, role, bucket)

## The model package for your Region

AWS Marketplace publishes one model package ARN per Region. The table below is filled in when the listing is published.

In [ ]:
# Filled in on publication: {region: model package ARN}.
MODEL_PACKAGE_ARNS = {
}

if region not in MODEL_PACKAGE_ARNS:
    raise SystemExit(
        f"No model package ARN for {region} yet. Supported: {sorted(MODEL_PACKAGE_ARNS) or 'none listed'}. "
        "Copy the ARN shown under 'Continue to configuration' on the listing into MODEL_PACKAGE_ARNS."
    )
model_package_arn = MODEL_PACKAGE_ARNS[region]
print(model_package_arn)

## Sample data

Public GRCh38 regions from Ensembl (see `data/sources.json`). Each file is a complete request body: `sequence` (A, C, G, T, N; 1,000 to 500,000 bp), an optional `sequence_name`, and optional `options`. Naming a sequence after its locus (for example `chr11:5225001-5235000`) and setting `options.shift_coordinates` to `"UCSC"` reports coordinates on the chromosome.

In [ ]:
DATA = Path("data")
samples = {p.stem: json.loads(p.read_text()) for p in sorted((DATA / "input").glob("*.json"))}
for name, body in samples.items():
    print(f"{name}: {len(body['sequence']):,} bp")

realtime = {n: b for n, b in samples.items() if REALTIME_MAX_BP is None or len(b["sequence"]) <= REALTIME_MAX_BP}
print("real-time samples:", sorted(realtime))

## Deploy a real-time endpoint

The model runs with network isolation, as every Marketplace model package does. This takes about 10 minutes.

In [ ]:
stamp = time.strftime("%Y%m%d-%H%M%S")
model_name = f"gi-annotation-{stamp}"
endpoint_name = f"gi-annotation-{stamp}"

sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={"ModelPackageName": model_package_arn},
    EnableNetworkIsolation=True,
)
sm.create_endpoint_config(
    EndpointConfigName=endpoint_name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InstanceType": INSTANCE,
        "InitialInstanceCount": 1,
    }],
)
sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_name)
sm.get_waiter("endpoint_in_service").wait(EndpointName=endpoint_name)
print("InService:", endpoint_name)

## Annotate the samples

The default JSON response has a `summary`, one entry per predicted transcript in `transcripts` (coordinates, strand, score, TSS and polyA positions, `transcript_type`, and `exons`, `introns` and `cds`), and the same result as GFF3 and BED text in `formats`.

In [ ]:
results = {}
for name, body in realtime.items():
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(body),
    )
    results[name] = json.loads(response["Body"].read())
    print(name, results[name]["summary"])

Ask for GFF3 directly with the `Accept` header, and write it to a file you can open in a genome browser:

In [ ]:
name = next(iter(realtime))
response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Accept="text/x-gff3",
    Body=json.dumps(realtime[name]),
)
gff3 = response["Body"].read().decode()
Path(f"{name}.gff3").write_text(gff3)
print("\n".join(gff3.splitlines()[:12]))

Compare with the published responses: the same transcripts, with the same coordinates and types.

In [ ]:
def structure(r):
    return sorted((t["start"], t["end"], t["strand"], t.get("transcript_type")) for t in r["transcripts"])

for name, r in results.items():
    expected = json.loads((DATA / "output" / f"{name}.json").read_text())
    print(f"{name}: {'same transcripts' if structure(r) == structure(expected) else 'DIFFERENT'} ({len(r['transcripts'])})")

## Batch transform

Each S3 object is one request body, passed whole to the model (`SplitType: None`), and each response comes back as its own `.out` object. Batch transform accepts every length up to 500,000 bp.

In [ ]:
for name, body in samples.items():
    s3.put_object(Bucket=bucket, Key=f"{prefix}/input/{name}.json", Body=json.dumps(body).encode())

job_name = f"gi-annotation-batch-{stamp}"
sm.create_transform_job(
    TransformJobName=job_name,
    ModelName=model_name,
    MaxConcurrentTransforms=1,
    BatchStrategy="SingleRecord",
    ModelClientConfig={"InvocationsTimeoutInSeconds": 600},  # TODO(measure): about twice the worst 500 kb time
    TransformInput={
        "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{bucket}/{prefix}/input/"}},
        "ContentType": "application/json",
        "SplitType": "None",
    },
    TransformOutput={"S3OutputPath": f"s3://{bucket}/{prefix}/output/", "AssembleWith": "None", "Accept": "application/json"},
    TransformResources={"InstanceType": INSTANCE, "InstanceCount": 1},
)
sm.get_waiter("transform_job_completed_or_stopped").wait(TransformJobName=job_name)
status = sm.describe_transform_job(TransformJobName=job_name)["TransformJobStatus"]
print(job_name, status)

for name in samples:
    obj = s3.get_object(Bucket=bucket, Key=f"{prefix}/output/{name}.json.out")
    r = json.loads(obj["Body"].read())
    print(name, r["summary"])

## Clean up

Deletes the endpoint (which stops its billing), the endpoint configuration and the model. The batch job stops billing on its own when it finishes.

In [ ]:
sm.delete_endpoint(EndpointName=endpoint_name)
sm.delete_endpoint_config(EndpointConfigName=endpoint_name)
sm.delete_model(ModelName=model_name)
print("deleted", endpoint_name)

To stop the subscription, open **Your Marketplace software** in the AWS Marketplace console, choose the product, and cancel the subscription.